# BDC 2026 - TTA Inference untuk 5 Model (TANPA training ulang)

Notebook ini **tidak melatih model apapun** -- murni memuat checkpoint yang sudah ada, lalu menjalankan
ulang inference (di data validasi/OOF maupun test) dengan **Test-Time Augmentation (TTA)**: tiap gambar
diprediksi dari beberapa versi (asli, flip horizontal, flip vertikal, rotasi 90 derajat), lalu
probabilitasnya dirata-rata. Biasanya memberi kenaikan kecil tapi konsisten dibanding prediksi 1x saja.

**5 model yang diproses**, checkpoint diambil dari 2 lokasi:

| Model | Lokasi checkpoint |
|---|---|
| ConvNeXtV2-Tiny | `.` (root, dari notebook stacking) |
| SigLIP2 ViT-B | `.` (root, dari notebook stacking) |
| ConvNeXtV2-Base | `checkpoints_large/` |
| ConvNeXtV2-Large | `checkpoints_large/` |
| SwinV2-Large | `checkpoints_large/` |

**Deteksi fold otomatis:** karena sempat tidak jelas apakah ConvNeXtV2-Tiny/SigLIP2 akhirnya dilatih
1 fold atau 5 fold (`use_kfold` sempat diubah), kode ini **mengecek langsung file checkpoint mana saja
yang ada di disk** (`{model}_fold0.pth`, `{model}_fold1.pth`, dst.) untuk tiap model, dan otomatis
menyesuaikan -- entah itu cuma 1 fold atau sampai 5 fold, tanpa perlu diberi tahu manual.

**Cache lama (tanpa TTA) TIDAK ditimpa** -- output disimpan dengan nama baru berakhiran `_tta`, supaya
masih bisa dibandingkan langsung skornya (dengan-TTA vs tanpa-TTA) sebelum memutuskan mana yang dipakai
di notebook penggabungan.

In [1]:
import os

import cv2
import numpy as np
import pandas as pd
import timm
import torch
from albumentations.pytorch import ToTensorV2
import albumentations as A
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
CONFIG = {
    "root": "BDC 2026",
    "num_workers": 0,
    "n_splits": 5,
    "seed": 42,
    # Sanity check jumlah data train -- ganti sesuai hasil pengecekan kamu kalau beda.
    # Ini murni buat mengingatkan/menghentikan lebih awal kalau folder train berubah lagi.
    "expected_n_train": 24954,
    "norm_stats": {
        "imagenet": {"mean": (0.485, 0.456, 0.406), "std": (0.229, 0.224, 0.225)},
        "siglip": {"mean": (0.5, 0.5, 0.5), "std": (0.5, 0.5, 0.5)},
    },
    "model_configs": [
        {"name": "convnextv2_tiny.fcmae_ft_in1k", "norm": "imagenet", "img_size": 224, "batch_size": 16, "checkpoint_dir": "."},
        {"name": "vit_base_patch16_siglip_224.v2_webli", "norm": "siglip", "img_size": 224, "batch_size": 16, "checkpoint_dir": "."},
        {"name": "convnextv2_base.fcmae_ft_in22k_in1k", "norm": "imagenet", "img_size": 224, "batch_size": 16, "checkpoint_dir": "checkpoints_large"},
        {"name": "convnextv2_large.fcmae_ft_in22k_in1k", "norm": "imagenet", "img_size": 224, "batch_size": 8, "checkpoint_dir": "checkpoints_large"},
        {"name": "swinv2_large_window12to16_192to256.ms_in22k_ft_in1k", "norm": "imagenet", "img_size": 256, "batch_size": 8, "checkpoint_dir": "checkpoints_large"},
    ],
    "oof_probs_tta_dir": "oof_probs_tta",
    "test_probs_tta_dir": "test_probs_tta",
}

## Deteksi Kelas & DataFrame

In [3]:
def get_class_order(config):
    train_dir = os.path.join(config["root"], "train")
    return sorted(os.listdir(train_dir))


def build_dataframe(train_dir, class_order):
    images, labels = [], []
    for c in class_order:
        folder = os.path.join(train_dir, c)
        for img in os.listdir(folder):
            images.append(os.path.join(folder, img))
            labels.append(c)
    df = pd.DataFrame({"image": images, "label": labels})

    label2id = {name: i for i, name in enumerate(class_order)}
    df["target"] = df["label"].map(label2id)
    return df


def build_test_dataframe(config):
    test_dir = os.path.join(config["root"], "test")
    test_images = sorted(
        os.listdir(test_dir),
        key=lambda x: int("".join(filter(str.isdigit, x)))
    )
    test_df = pd.DataFrame({"image": test_images})
    test_df["id"] = test_df["image"].apply(lambda x: int("".join(filter(str.isdigit, x))))
    return test_df

## Dataset

In [4]:
class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = cv2.imread(row["image"])
        if image is None:
            image = np.array(Image.open(row["image"]).convert("RGB"))
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, row["target"]


class TestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.loc[idx, "image"]
        image_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(image_path)
        if image is None:
            image = np.array(Image.open(image_path).convert("RGB"))
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, image_name

## Transform (cuma butuh versi validasi/inference, tidak ada augmentasi training)

In [5]:
def get_transforms(img_size, norm_type, config):
    stats = config["norm_stats"][norm_type]
    valid_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=stats["mean"], std=stats["std"]),
        ToTensorV2(),
    ])
    return valid_transform

## K-Fold Split (harus identik dgn yang dipakai waktu training, biar valid_idx cocok)

In [6]:
def get_skf(config):
    return StratifiedKFold(n_splits=config["n_splits"], shuffle=True, random_state=config["seed"])

## Device

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 3060


## TTA

4 variasi per gambar: asli, flip horizontal, flip vertikal, rotasi 90 derajat. Probabilitas softmax
dari keempatnya dirata-rata.

In [8]:
def predict_tta(model, images):
    variants = [
        images,
        torch.flip(images, dims=[3]),            # horizontal flip
        torch.flip(images, dims=[2]),             # vertical flip
        torch.rot90(images, k=1, dims=[2, 3]),    # rotate 90 derajat
    ]
    probs_sum = None
    for v in variants:
        outputs = model(v)
        probs = torch.softmax(outputs, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(variants)

## Pipeline TTA Inference per Model

`available_folds` dideteksi otomatis dari checkpoint yang benar-benar ada di disk (`{model}_fold0.pth`,
`{model}_fold1.pth`, dst.) -- bisa 1 fold atau sampai 5 fold, kode ini menyesuaikan otomatis. OOF diambil
dari data validasi tiap fold yang punya checkpoint; test probs dirata-rata (bagging) dari semua fold yang
tersedia.

In [9]:
def run_tta_inference(model_cfg, df, test_df, skf, device, config):
    model_name = model_cfg["name"]
    norm_type = model_cfg["norm"]
    img_size = model_cfg["img_size"]
    batch_size = model_cfg["batch_size"]
    checkpoint_dir = model_cfg["checkpoint_dir"]
    safe_name = model_name.replace("/", "_")

    valid_transform = get_transforms(img_size, norm_type, config)

    n_train = len(df)
    n_classes = df["target"].nunique()
    n_test = len(test_df)

    all_splits = list(skf.split(df, df["target"]))

    # Deteksi fold checkpoint mana saja yang benar-benar ada di disk
    available_folds = []
    for fold in range(len(all_splits)):
        ckpt_path = os.path.join(checkpoint_dir, f"{safe_name}_fold{fold}.pth")
        if os.path.exists(ckpt_path):
            available_folds.append(fold)

    if not available_folds:
        raise FileNotFoundError(
            f"Tidak ada checkpoint ditemukan untuk {model_name} di '{checkpoint_dir}/' "
            f"(dicari: {safe_name}_fold0.pth, {safe_name}_fold1.pth, dst.)"
        )

    print(f"\n{'=' * 60}")
    print(f"{model_name}")
    print(f"{len(available_folds)} fold checkpoint ditemukan: {available_folds}")
    print(f"{'=' * 60}")

    oof_probs = np.full((n_train, n_classes), np.nan)
    test_probs_per_fold = []

    test_dataset = TestDataset(test_df, image_dir=os.path.join(config["root"], "test"), transform=valid_transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=config["num_workers"], pin_memory=True)

    for fold in available_folds:
        train_idx, valid_idx = all_splits[fold]
        valid_df_fold = df.iloc[valid_idx].reset_index(drop=True)
        valid_dataset = WasteDataset(valid_df_fold, transform=valid_transform)
        valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=config["num_workers"], pin_memory=True)

        ckpt_path = os.path.join(checkpoint_dir, f"{safe_name}_fold{fold}.pth")

        # img_size cuma di-pass eksplisit untuk SwinV2 (window-based, resolusi wajib
        # sesuai training). ConvNeXt/SigLIP fully-convolutional/flexible -- tidak perlu.
        if "swinv2" in model_name:
            model = timm.create_model(model_name, pretrained=False, num_classes=n_classes, img_size=img_size)
        else:
            model = timm.create_model(model_name, pretrained=False, num_classes=n_classes)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.to(device)
        model.eval()

        print(f"\nFold {fold}: TTA di {len(valid_df_fold)} data validasi (OOF)...")
        fold_oof_probs = []
        with torch.no_grad():
            for images, _ in valid_loader:
                images = images.to(device)
                probs = predict_tta(model, images)
                fold_oof_probs.append(probs.cpu().numpy())
        oof_probs[valid_idx] = np.concatenate(fold_oof_probs, axis=0)

        print(f"Fold {fold}: TTA di {n_test} data test...")
        fold_test_probs = []
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)
                probs = predict_tta(model, images)
                fold_test_probs.append(probs.cpu().numpy())
        test_probs_per_fold.append(np.concatenate(fold_test_probs, axis=0))

        del model
        torch.cuda.empty_cache()

    valid_mask = ~np.isnan(oof_probs).any(axis=1)
    oof_preds = oof_probs[valid_mask].argmax(axis=1)
    oof_f1 = f1_score(df.loc[valid_mask, "target"], oof_preds, average="macro")
    print(f"\n>> {model_name} | OOF macro F1 DENGAN TTA ({valid_mask.sum()} baris tervalidasi): {oof_f1:.4f}")

    test_probs = np.mean(test_probs_per_fold, axis=0)  # bagging antar fold yang tersedia

    os.makedirs(config["oof_probs_tta_dir"], exist_ok=True)
    os.makedirs(config["test_probs_tta_dir"], exist_ok=True)
    oof_path = os.path.join(config["oof_probs_tta_dir"], f"{safe_name}_oof_tta.npy")
    test_path = os.path.join(config["test_probs_tta_dir"], f"{safe_name}_test_tta.npy")
    np.save(oof_path, oof_probs)
    np.save(test_path, test_probs)
    print(f"Cache TTA disimpan: {oof_path}, {test_path}")

    return oof_probs, test_probs, valid_mask

## Run

Tidak ada training di sini -- cuma load checkpoint & inference ulang dengan TTA untuk kelima model.
Bisa lebih cepat dari training (apalagi model yang cuma 1 fold), tapi model Large tetap butuh waktu
untuk 2x forward pass penuh (OOF + test) x 4 augmentasi TTA.

In [10]:
class_order = get_class_order(CONFIG)
print("Label mapping:", {name: i for i, name in enumerate(class_order)})

train_dir = os.path.join(CONFIG["root"], "train")
df = build_dataframe(train_dir, class_order)
test_df = build_test_dataframe(CONFIG)

print(f"Total data train saat ini: {len(df)} baris (ekspektasi: {CONFIG['expected_n_train']})")
assert len(df) == CONFIG["expected_n_train"], (
    f"Jumlah data train ({len(df)}) tidak sesuai ekspektasi ({CONFIG['expected_n_train']}). "
    f"Cek lagi apakah folder train/ berubah -- lihat catatan di judul notebook ini."
)

skf = get_skf(CONFIG)

oof_tta_list = []
test_tta_list = []
valid_mask_tta_list = []
model_names = []

for model_cfg in CONFIG["model_configs"]:
    oof_p, test_p, valid_mask = run_tta_inference(model_cfg, df, test_df, skf, device, CONFIG)
    oof_tta_list.append(oof_p)
    test_tta_list.append(test_p)
    valid_mask_tta_list.append(valid_mask)
    model_names.append(model_cfg["name"])

print("\n" + "=" * 60)
print("SEMUA MODEL SELESAI TTA INFERENCE")
print("=" * 60)
for name in model_names:
    print(f"  - {name}")
print(f"\nCache baru (dengan TTA): {CONFIG['oof_probs_tta_dir']}/, {CONFIG['test_probs_tta_dir']}/")
print("Cache lama (tanpa TTA) tidak diubah -- masih ada untuk dibandingkan.")
print("\nLangkah selanjutnya: update path cache di notebook penggabungan (combine_5model_stacking.ipynb)")
print("supaya mengarah ke folder '_tta' ini, lalu jalankan ulang untuk lihat submission barunya.")

Label mapping: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
Total data train saat ini: 24954 baris (ekspektasi: 24954)

convnextv2_tiny.fcmae_ft_in1k
5 fold checkpoint ditemukan: [0, 1, 2, 3, 4]

Fold 0: TTA di 4991 data validasi (OOF)...
Fold 0: TTA di 1458 data test...

Fold 1: TTA di 4991 data validasi (OOF)...
Fold 1: TTA di 1458 data test...

Fold 2: TTA di 4991 data validasi (OOF)...
Fold 2: TTA di 1458 data test...

Fold 3: TTA di 4991 data validasi (OOF)...
Fold 3: TTA di 1458 data test...

Fold 4: TTA di 4990 data validasi (OOF)...
Fold 4: TTA di 1458 data test...

>> convnextv2_tiny.fcmae_ft_in1k | OOF macro F1 DENGAN TTA (24954 baris tervalidasi): 0.9875
Cache TTA disimpan: oof_probs_tta\convnextv2_tiny.fcmae_ft_in1k_oof_tta.npy, test_probs_tta\convnextv2_tiny.fcmae_ft_in1k_test_tta.npy

vit_base_patch16_siglip_224.v2_webli
5 fold checkpoint ditemukan: [0, 1, 2, 3, 4]

Fold 0: TTA di 4991 data validasi (OOF)...
Fold 0: TTA di 1458 data test...

Fold 1: TTA di 4991 